In [6]:
import pandas as pd
import numpy as np

# Read the dataset
df = pd.read_csv(r'D:\Swiggy\swiggy.csv')

# 1. Data Understanding
print("Initial shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

# 2. Remove Duplicates
df_cleaned = df.drop_duplicates()
print("\nShape after removing duplicates:", df_cleaned.shape)

# 3. Handle Missing Values
# Option 1: Drop rows with missing values
df_cleaned = df_cleaned.dropna()

# Option 2: Impute missing values
# For numerical columns
numeric_columns = df_cleaned.select_dtypes(include=[np.number]).columns
df_cleaned[numeric_columns] = df_cleaned[numeric_columns].fillna(df_cleaned[numeric_columns].mean())

# For categorical columns
categorical_columns = df_cleaned.select_dtypes(include=['object']).columns
df_cleaned[categorical_columns] = df_cleaned[categorical_columns].fillna(df_cleaned[categorical_columns].mode().iloc[0])

# 4. Save cleaned dataset
df_cleaned.to_csv('cleaned_data.csv', index=False)
print("\nCleaned data saved successfully!")


Initial shape: (148541, 11)

Missing values:
 id                0
name             86
city              0
rating           86
rating_count     86
cost            131
cuisine          99
lic_no          229
link              0
address          86
menu              0
dtype: int64

Duplicate rows: 0

Shape after removing duplicates: (148541, 11)

Cleaned data saved successfully!


In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import pickle

# 1. Load the cleaned dataset
df = pd.read_csv('cleaned_data.csv')

# 2. Initialize dictionary to store label encoders
label_encoders = {}

# 3. Process each categorical column separately using Label Encoding
categorical_columns = ['name', 'city', 'cuisine']
for column in categorical_columns:
    # Create a label encoder for each column
    label_encoders[column] = LabelEncoder()
    # Fit and transform the column
    df[f"{column}_encoded"] = label_encoders[column].fit_transform(df[column])

# 4. Drop original categorical columns
df_encoded = df.drop(columns=categorical_columns)

# 5. Save the label encoders
with open('label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

# 6. Save the encoded dataset
df_encoded.to_csv('encoded_data.csv', index=False)

# 7. Print information about the encoding
print("\nEncoding Summary:")
for column in categorical_columns:
    n_categories = len(label_encoders[column].classes_)
    print(f"{column}: {n_categories} unique categories")

print("\nOriginal shape:", df.shape)
print("Encoded shape:", df_encoded.shape)
print("Encoders saved: label_encoders.pkl")
print("Encoded data saved: encoded_data.csv")

# 8. Optional: Print sample of encoded data
print("\nSample of encoded data:")
print(df_encoded.head())



Encoding Summary:
name: 112683 unique categories
city: 821 unique categories
cuisine: 2131 unique categories

Original shape: (148255, 14)
Encoded shape: (148255, 11)
Encoders saved: label_encoders.pkl
Encoded data saved: encoded_data.csv

Sample of encoded data:
       id rating     rating_count   cost          lic_no  \
0  567335     --  Too Few Ratings  ₹ 200  22122652000138   
1  531342    4.4      50+ ratings  ₹ 200  12117201000112   
2  158203    3.8     100+ ratings  ₹ 100  22121652000190   
3  187912    3.7      20+ ratings  ₹ 250  22119652000167   
4  543530     --  Too Few Ratings  ₹ 250  12122201000053   

                                                link  \
0  https://www.swiggy.com/restaurants/ab-foods-po...   
1  https://www.swiggy.com/restaurants/janta-sweet...   
2  https://www.swiggy.com/restaurants/theka-coffe...   
3  https://www.swiggy.com/restaurants/singh-hut-n...   
4  https://www.swiggy.com/restaurants/grill-maste...   

                                     

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
import pickle
from collections import defaultdict

# Load and prepare data 
encoded_df = pd.read_csv('encoded_data.csv')
original_df = pd.read_csv('cleaned_data.csv')

def clean_and_prepare_data(df):
    df_clean = df.copy()
    columns_to_use = ['id', 'name_encoded', 'city_encoded', 'cuisine_encoded']
    df_clean['rating'] = pd.to_numeric(df_clean['rating'].replace('--', np.nan), errors='coerce')
    columns_to_use.append('rating')
    df_clean = df_clean[columns_to_use]
    df_clean = df_clean.fillna(df_clean.mean())
    return df_clean

# Clean and scale data
encoded_df_clean = clean_and_prepare_data(encoded_df)
scaler = StandardScaler()
scaled_features = scaler.fit_transform(encoded_df_clean)

# Save scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# ================ Method 1: MiniBatch K-Means Clustering ================
def create_cluster_based_recommender(n_clusters=100, batch_size=1000):
    # Use MiniBatchKMeans instead of regular KMeans
    kmeans = MiniBatchKMeans(
        n_clusters=n_clusters,
        batch_size=batch_size,
        random_state=42
    )
    cluster_labels = kmeans.fit_predict(scaled_features)
    
    # Create cluster to restaurant mapping
    cluster_to_restaurants = defaultdict(list)
    for idx, label in enumerate(cluster_labels):
        cluster_to_restaurants[label].append(idx)
    
    # Save the model and mapping
    with open('minibatch_kmeans_model.pkl', 'wb') as f:
        pickle.dump((kmeans, cluster_to_restaurants), f)
    
    def get_recommendations(restaurant_id, n_recommendations=5):
        try:
            # Get restaurant's cluster
            restaurant_features = scaled_features[restaurant_id].reshape(1, -1)
            restaurant_cluster = kmeans.predict(restaurant_features)[0]
            
            # Get other restaurants in the same cluster
            cluster_restaurants = cluster_to_restaurants[restaurant_cluster]
            
            # Filter out the input restaurant
            other_restaurants = [idx for idx in cluster_restaurants if idx != restaurant_id]
            
            # Sort by rating (if available)
            recommendations_df = original_df.iloc[other_restaurants].copy()
            recommendations_df['rating'] = pd.to_numeric(recommendations_df['rating'], errors='coerce')
            recommendations_df = recommendations_df.sort_values('rating', ascending=False)
            
            return recommendations_df.head(n_recommendations)
            
        except Exception as e:
            print(f"Error getting recommendations: {str(e)}")
            return pd.DataFrame()
    
    return get_recommendations

# ================ Method 2: Nearest Neighbors in Batch ================
def get_nearest_neighbors(restaurant_id, n_recommendations=5):
    """Get recommendations using batch processing for memory efficiency"""
    target_features = scaled_features[restaurant_id]
    batch_size = 1000
    nearest_neighbors = []
    
    # Process in batches
    for i in range(0, len(scaled_features), batch_size):
        batch = scaled_features[i:i + batch_size]
        # Calculate distances for this batch
        distances = np.linalg.norm(batch - target_features, axis=1)
        # Get top indices for this batch
        batch_indices = np.argsort(distances)[:n_recommendations]
        batch_distances = distances[batch_indices]
        
        # Add to overall results
        for idx, dist in zip(batch_indices + i, batch_distances):
            nearest_neighbors.append((idx, dist))
        
        # Sort and keep only top N
        nearest_neighbors.sort(key=lambda x: x[1])
        nearest_neighbors = nearest_neighbors[:n_recommendations]
    
    # Get indices of nearest neighbors
    neighbor_indices = [idx for idx, _ in nearest_neighbors]
    return original_df.iloc[neighbor_indices]

# Initialize recommender
recommender = create_cluster_based_recommender()

def get_recommendations(restaurant_id, n_recommendations=5):
    """Get recommendations using both methods"""
    print(f"\nRecommendations for Restaurant ID: {restaurant_id}")
    
    # Print original restaurant details
    print("\nOriginal Restaurant:")
    print(original_df.iloc[restaurant_id][['name', 'city', 'cuisine', 'rating']])
    
    print("\n1. Cluster-Based Recommendations:")
    cluster_recs = recommender(restaurant_id, n_recommendations)
    if not cluster_recs.empty:
        print(cluster_recs[['name', 'city', 'cuisine', 'rating']])
    
    print("\n2. Nearest Neighbor Recommendations:")
    nn_recs = get_nearest_neighbors(restaurant_id, n_recommendations)
    if not nn_recs.empty:
        print(nn_recs[['name', 'city', 'cuisine', 'rating']])

# Helper function to find restaurant by name
def find_restaurant_by_name(name):
    matches = original_df[original_df['name'].str.contains(name, case=False, na=False)]
    if not matches.empty:
        return matches.index[0]
    return None

# Example usage
print("\nTesting recommendation system:")
test_id = 1  
get_recommendations(test_id, n_recommendations=3)



Testing recommendation system:

Recommendations for Restaurant ID: 1

Original Restaurant:
name       Janta Sweet House
city                  Abohar
cuisine        Sweets,Bakery
rating                   4.4
Name: 1, dtype: object

1. Cluster-Based Recommendations:
                       name                     city          cuisine  rating
82385        Agrawal Sweets       Bhawar Kuan,Indore    Sweets,Bakery     5.0
64613      Anmol Bhel House                   Gondal           Snacks     4.8
74611  Govi's Organic Foods  Banjara Hills,Hyderabad  Sweets,Desserts     4.8

2. Nearest Neighbor Recommendations:
                           name           city        cuisine rating
1             Janta Sweet House         Abohar  Sweets,Bakery    4.4
6998     Kaka Na Bhungra Bataka          Anand  Snacks,Indian    4.4
43149  Kovilpatti Murukku Kadai  Adyar,Chennai  Snacks,Sweets    4.4
